In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
# Read the dataset Q1_data.csv using read_csv()
food_path = os.path.join(path, 'Q1_data.csv')
df_food = pd.read_csv(food_path)

print(f"Dataset shape: {df_food.shape}")

In [ ]:
# Task 2: Write your code here:
# Inspect the first few rows using head()
df_food.head()

In [ ]:
# Task 3: Write your code here:
# Display dataset information using info()
df_food.info()

In [ ]:
# Task 4: Write your code here:
# Show statistical description using describe()
df_food.describe()

In [ ]:
# Task 5: Write your code here:
# Plot the target distribution (delivery_time)
plt.figure(figsize=(10, 5))
plt.hist(df_food['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Delivery Time Distribution')
plt.xlabel('Delivery Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
# Drop the 'Order_ID' column from the data
cols = ['Order_ID', 'Distance_km', 'Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type', 'Preparation_Time_min', 'Courier_Experience_yrs', 'Delivery_Time']
df_clean = df_food[cols].copy()

print(f"Before: {df_clean.shape}")
df_clean = df_clean.drop(columns="Order_ID", axis=1)
print(f"After dropping Order ID: {df_clean.shape}")

In [ ]:
# Missing values
print("Missing values:")
print(df_food.isnull().sum())

In [ ]:
# Task 2: Write your code here:
# Handle missing values appropriately (Hint: I guess you want to have a closer look at the columns with missing values :) )

# Fill categorical columns with 'unknown' - missing likely means "not specified"
for col in ['Weather', 'Traffic_Level', 'Time_of_Day']:
    df_clean[col] = df_clean[col].fillna('unknown')

# Fill Delivery_Time and Courier_Experience_yrs with mean
df_clean['Courier_Experience_yrs'] = df_clean['Courier_Experience_yrs'].fillna(df_clean['Courier_Experience_yrs'].mean())
df_clean['Delivery_Time'] = df_clean['Delivery_Time'].fillna(df_clean['Delivery_Time'].mean())

print("Missing values after cleaning:")
print(df_clean.isnull().sum())

In [ ]:
# Task 3: Write your code here:
# Check and remove duplicates if any exist

def check_duplicates(df):
  duplicates = df_clean.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)

In [ ]:
# Task 4: Write your code here:
# Encode categorical variables if needed (Bonus if used One Hot Encoding)
# Encode categorical columns - converts text to integers
categorical_cols = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type']
for col in categorical_cols:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))

df_clean.head()


In [ ]:
# Task 5: Write your code here:
# Apply feature scaling for all features (Use StandardScaler)
numerical_cols = df_clean.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df_clean[numerical_cols] = scaler.fit_transform(df_clean[numerical_cols])
df_clean.head()

In [ ]:
# Task 6: Write your code here:
# Check for target imbalance and state if it is imbalanced or not (keep this cell empty if not needed)

In [ ]:
# Task 1: Write your code here:
# Split the dataset into features (X) and target (y)
feature_cols = ['Distance_km', 'Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type', 'Preparation_Time_min', 'Courier_Experience_yrs']
X = df_clean[feature_cols]
y = df_clean['Delivery_Time']


In [ ]:
# Task 2,3,4,5: Write your code here:
# (2) Use the correct split: KFold OR StratifiedKFold
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

# Scale features - fit on train, transform both
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nScaled ranges - Min: {X_train_scaled.min():.2f}, Max: {X_train_scaled.max():.2f}")
pd.DataFrame(X_train_scaled, columns=X_train.columns).head(3)

# (3) Train a RandomForest model
model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
model.fit(X_train_scaled, y_train)
print("Model trained!")

# (4) Evaluate using MAE (Mean Absolute Error) ONLY
y_pred = model.predict(X_test_scaled)

mae = mean_absolute_error(y_test, y_pred)

# (5) Print the averaged score across all folds
print(f"MAE: {mae:,.2f} minutes")


In [ ]:
# Task 1: Write your code here:
# Plot feature importance from your trained model
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()


In [ ]:
# Task 2: Write your code here:
# Plot predicted delivery time histogram
# Plot Predictions vs Ground Truth
plt.figure(figsize=(10, 5))
plt.hist(df_clean['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Predicted Delivery Time Distribution')
plt.xlabel('Predicted Delivery Time')
plt.ylabel('Frequency')
plt.show()


In [ ]:
# Task Bonus: Write your code here:
